<a href="https://colab.research.google.com/github/gsg2144-svg/gsg2144-svg.github.io/blob/main/Geopolitical_Risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Country Risk Scoring Tool**

This tool automatically generates a country risk assessment by using an AI model (GPT-5) to analyze a given country’s economic and political situation. When you enter a country name, the model draws on its general knowledge to evaluate four key pillars: macroeconomic conditions, political and institutional stability, external and financial vulnerability, and structural and transition risks. To do this, it implicitly considers variables such as GDP growth, inflation, public debt, fiscal balance, and monetary stability (macroeconomic); governance quality, policy predictability, corruption, and social unrest (political); current account balance, external debt, foreign reserves, exchange rate stability, and banking sector health (external/financial); and longer-term factors like economic diversification, commodity dependence, demographics, climate vulnerability, and energy transition exposure (structural). It then assigns a risk score to each pillar, combines them into an overall country risk score, and produces a clear summary of the main risks, strengths, and short-term outlook, similar to how an analyst at a bank or investment fund would assess country risk.

In [11]:
from __future__ import annotations

import json
from dataclasses import dataclass
from typing import Dict, Any

from google.colab import userdata
from openai import OpenAI
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets


# ============================================================
# 1. OPENAI CLIENT
# ============================================================

client = OpenAI(api_key=userdata.get("openai"))
MODEL_NAME = "gpt-5"


# ============================================================
# 2. OUTPUT STRUCTURE
# ============================================================

COUNTRY_RISK_SCHEMA = {
    "type": "object",
    "properties": {
        "country": {"type": "string"},
        "executive_summary": {"type": "string"},
        "macroeconomic_risk": {
            "type": "object",
            "properties": {
                "score": {"type": "number"},
                "summary": {"type": "string"},
            },
            "required": ["score", "summary"],
            "additionalProperties": False,
        },
        "political_institutional_risk": {
            "type": "object",
            "properties": {
                "score": {"type": "number"},
                "summary": {"type": "string"},
            },
            "required": ["score", "summary"],
            "additionalProperties": False,
        },
        "external_financial_risk": {
            "type": "object",
            "properties": {
                "score": {"type": "number"},
                "summary": {"type": "string"},
            },
            "required": ["score", "summary"],
            "additionalProperties": False,
        },
        "structural_transition_risk": {
            "type": "object",
            "properties": {
                "score": {"type": "number"},
                "summary": {"type": "string"},
            },
            "required": ["score", "summary"],
            "additionalProperties": False,
        },
        "key_risk_drivers": {
            "type": "array",
            "items": {"type": "string"},
        },
        "key_strengths": {
            "type": "array",
            "items": {"type": "string"},
        },
        "outlook_12_24_months": {"type": "string"},
        "final_risk_score": {"type": "number"},
        "risk_label": {"type": "string"},
    },
    "required": [
        "country",
        "executive_summary",
        "macroeconomic_risk",
        "political_institutional_risk",
        "external_financial_risk",
        "structural_transition_risk",
        "key_risk_drivers",
        "key_strengths",
        "outlook_12_24_months",
        "final_risk_score",
        "risk_label",
    ],
    "additionalProperties": False,
}


# ============================================================
# 3. MODEL CALL
# ============================================================

SYSTEM_PROMPT = """
You are a sovereign risk analyst.

Your job is to produce a full country risk assessment for a given country.

Instructions:
- Write a concise but substantive sovereign/country risk summary.
- Assess four pillars:
  1. Macroeconomic risk
  2. Political and institutional risk
  3. External and financial risk
  4. Structural and transition risk
- Score each pillar from 0 to 100, where:
  - 0 = very low risk
  - 25 = low risk
  - 50 = moderate risk
  - 75 = high risk
  - 100 = very high risk
- Then produce a final country risk score from 0 to 100.
- Assign one label only:
  - Very Low Risk
  - Low Risk
  - Moderate Risk
  - High Risk
  - Very High Risk

Scoring logic:
- final_risk_score should reflect the four pillar scores combined.
- Be conservative, balanced, and analytic.
- Do not use bullet fragments in summaries. Write clear prose.
- Return valid JSON only.
""".strip()


def extract_response_text(response: Any) -> str:
    if hasattr(response, "output_text") and response.output_text:
        return response.output_text

    texts = []
    try:
        for item in response.output:
            if getattr(item, "type", None) == "message":
                for content in item.content:
                    if hasattr(content, "text") and content.text:
                        texts.append(content.text)
        return "\n".join(texts).strip()
    except Exception:
        raise ValueError("Could not extract text from model response.")


def get_country_risk_report(country: str) -> Dict[str, Any]:
    user_prompt = f"""
Country to assess: {country}

Produce a full sovereign country risk assessment for this country.
You should infer the likely current risk profile based on general macroeconomic,
political, external, and structural characteristics.

Return JSON only.
""".strip()

    response = client.responses.create(
        model=MODEL_NAME,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "country_risk_report",
                "schema": COUNTRY_RISK_SCHEMA,
                "strict": True,
            }
        },
    )

    raw_text = extract_response_text(response)
    return json.loads(raw_text)


# ============================================================
# 4. FORMATTER
# ============================================================

def format_country_risk_markdown(data: Dict[str, Any]) -> str:
    md = []

    md.append(f"# Country Risk Report: {data['country']}")
    md.append("")
    md.append(f"**Final Risk Score:** {data['final_risk_score']:.1f}/100")
    md.append(f"**Risk Label:** {data['risk_label']}")
    md.append("")
    md.append("## Executive Summary")
    md.append(data["executive_summary"])
    md.append("")

    md.append("## Pillar Scores")
    md.append(f"- **Macroeconomic Risk:** {data['macroeconomic_risk']['score']:.1f}")
    md.append(f"- **Political and Institutional Risk:** {data['political_institutional_risk']['score']:.1f}")
    md.append(f"- **External and Financial Risk:** {data['external_financial_risk']['score']:.1f}")
    md.append(f"- **Structural and Transition Risk:** {data['structural_transition_risk']['score']:.1f}")
    md.append("")

    md.append("## Macroeconomic Risk")
    md.append(data["macroeconomic_risk"]["summary"])
    md.append("")

    md.append("## Political and Institutional Risk")
    md.append(data["political_institutional_risk"]["summary"])
    md.append("")

    md.append("## External and Financial Risk")
    md.append(data["external_financial_risk"]["summary"])
    md.append("")

    md.append("## Structural and Transition Risk")
    md.append(data["structural_transition_risk"]["summary"])
    md.append("")

    md.append("## Key Risk Drivers")
    for item in data["key_risk_drivers"]:
        md.append(f"- {item}")
    md.append("")

    md.append("## Key Strengths")
    for item in data["key_strengths"]:
        md.append(f"- {item}")
    md.append("")

    md.append("## 12–24 Month Outlook")
    md.append(data["outlook_12_24_months"])

    return "\n".join(md)


# ============================================================
# 5. NOTEBOOK UI
# ============================================================

country_widget = widgets.Text(
    value="",
    placeholder="Enter country name",
    description="Country:",
    layout=widgets.Layout(width="700px")
)

run_button = widgets.Button(
    description="Run Country Risk",
    button_style="primary"
)

output = widgets.Output()


def on_run_clicked(_):
    with output:
        clear_output()

        country = country_widget.value.strip()

        if not country:
            display(Markdown("**Error:** Please enter a country name."))
            return

        display(Markdown(f"Running country risk report for **{country}** ..."))

        try:
            result = get_country_risk_report(country)
            clear_output()
            display(Markdown(format_country_risk_markdown(result)))
        except Exception as e:
            clear_output()
            display(Markdown(f"**Error:** `{str(e)}`"))


run_button.on_click(on_run_clicked)

display(
    widgets.VBox([
        widgets.HTML("<h2>Country Risk Scoring Tool</h2>"),
        country_widget,
        run_button,
        output
    ])
)